In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
FILE_PATH = "data/361_1day_FPB.csv"

C_OUT = 415            # outdoor CO2 ppm
G = 0.005              # L/s per person (typical sedentary)
CFM_TO_LPS = 0.471947  # conversion

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(FILE_PATH)

# Convert datetime
df['dateTime'] = pd.to_datetime(df['dateTime'])

# =========================
# FILTER DATA
# =========================
co2 = df[df['pointDisplayName'] == 'Zone CO2'].copy()
flow = df[df['pointDisplayName'] == 'Discharge Air Flow'].copy()

co2 = co2.set_index('dateTime').sort_index()
flow = flow.set_index('dateTime').sort_index()

# =========================
# RESAMPLE (5-minute average)
# =========================
co2_res = co2['value'].resample('5min').mean()
flow_res = flow['value'].resample('5min').mean()

merged = pd.concat([co2_res, flow_res], axis=1)
merged.columns = ['co2', 'cfm']
merged = merged.dropna()

# =========================
# CALCULATE OCCUPANCY
# =========================
merged['delta'] = merged['co2'] - C_OUT

# Convert airflow
merged['Lps'] = merged['cfm'] * CFM_TO_LPS

# Occupancy formula
merged['people'] = (merged['Lps'] * merged['delta']) / (1e6 * G)

# Remove negatives
merged['people'] = merged['people'].clip(lower=0)

# Smooth (optional but recommended)
merged['people_smooth'] = merged['people'].rolling(window=3, min_periods=1).mean()

# =========================
# PLOTTING
# =========================
plt.figure(figsize=(14, 8))

# CO2
plt.subplot(3, 1, 1)
plt.plot(merged.index, merged['co2'])
plt.title("CO₂ (ppm)")
plt.ylabel("ppm")

# Airflow
plt.subplot(3, 1, 2)
plt.plot(merged.index, merged['cfm'])
plt.title("Airflow (CFM)")
plt.ylabel("CFM")

# Occupancy
plt.subplot(3, 1, 3)
plt.plot(merged.index, merged['people_smooth'])
plt.title("Estimated Occupancy")
plt.ylabel("People")
plt.xlabel("Time")

plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '361_1day_FPB.csv'